# Phase 2.4 — Pandas & Data Handling

Applies core Pandas operations to the project's NSE company-master and historical market datasets: loading, inspection, filtering, sorting, aggregation, merging, datetime operations, returns, and rolling statistics.

This phase is for data-handling learning and validation. Final investment targets and ML features are created in later phases.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/stock-investment-ml")
PROCESSED = ROOT / "data/processed/nse"
company_files = sorted(PROCESSED.glob("*.parquet"))
market_files = sorted((PROCESSED / "market").glob("*.parquet"))

if not company_files or not market_files:
    raise FileNotFoundError("Required company-master or market Parquet dataset not found.")

company_df = pd.read_parquet(company_files[0])
market_df = pd.read_parquet(market_files[-1])

if company_df.empty or market_df.empty:
    raise ValueError("Loaded dataset is empty.")

# Current market schema uses instrument_name rather than company_name.
name_col = "company_name" if "company_name" in market_df.columns else "instrument_name"
market_df = market_df.rename(columns={name_col: "company_name"})
market_df["trade_date"] = pd.to_datetime(market_df["trade_date"])

print(f"Company master: {company_df.shape}")
print(f"Market data   : {market_df.shape}")
print(f"Market range  : {market_df['trade_date'].min().date()} → {market_df['trade_date'].max().date()}")

In [ ]:
print("=== COLUMNS ===")
print("Company:", company_df.columns.tolist())
print("Market :", market_df.columns.tolist())

print("\n=== TYPES ===")
print(market_df.dtypes)

print("\n=== DIMENSIONS ===")
print(f"Company: {len(company_df):,} × {company_df.shape[1]}")
print(f"Market : {len(market_df):,} × {market_df.shape[1]}")

print("\n=== SUMMARY ===")
display(market_df.describe())

print("\n=== MISSING VALUES ===")
display(market_df.isna().sum().to_frame("missing_count"))

In [ ]:
print("=== CATEGORICAL VALUES ===")
for col in ["series", "instrument_type", "segment"]:
    if col in market_df:
        print(f"\n{col}")
        print(market_df[col].value_counts(dropna=False))

dates = market_df["trade_date"].value_counts().sort_index()
print("\n=== TRADING DATES ===")
print(f"Trading days: {len(dates)}")
print(f"Range       : {dates.index.min().date()} → {dates.index.max().date()}")
display(dates.rename("records"))

latest_date = market_df["trade_date"].max()
latest = market_df[market_df["trade_date"].eq(latest_date)]
        
print(f"\n=== LATEST DATE: {latest_date.date()} ===")
print(f"Rows: {len(latest):,}")
display(latest[["nse_symbol", "company_name", "close"]].head())
        
print("\n=== HIGHEST CLOSING PRICES ===")
display(market_df.nlargest(10, "close")[["nse_symbol", "company_name", "trade_date", "close"]])

In [ ]:
duplicate_count = market_df.duplicated(["trade_date", "instrument_id"]).sum()
print("=== DATA QUALITY ===")
print(f"Duplicate company/date rows: {duplicate_count:,}")
print("Missing values:")
print(market_df.isna().sum()[lambda x: x.gt(0)])

daily_summary = market_df.groupby("trade_date").agg(
    companies=("instrument_id", "nunique"),
    total_volume=("volume", "sum"),
    total_turnover=("turnover", "sum"),
    total_trades=("number_of_trades", "sum")
)

company_summary = market_df.groupby(["nse_symbol", "company_name"]).agg(
    trading_days=("trade_date", "nunique"),
    average_close=("close", "mean"),
    average_volume=("volume", "mean"),
    average_turnover=("turnover", "mean")
).sort_values("average_turnover", ascending=False)

print("\n=== DAILY SUMMARY ===")
display(daily_summary.head())
print("=== COMPANY SUMMARY ===")
display(company_summary.head(10))

In [ ]:
merged_df = market_df.merge(
    company_df[["isin", "paid_up_value", "market_lot", "face_value"]],
    on="isin", how="left", validate="many_to_one"
)

unmatched = merged_df[["paid_up_value", "market_lot", "face_value"]].isna().any(axis=1).sum()

print("=== MERGE VALIDATION ===")
print(f"Market rows : {len(market_df):,}")
print(f"Merged rows : {len(merged_df):,}")
print(f"Unmatched   : {unmatched:,}")
display(merged_df[["nse_symbol", "company_name", "trade_date", "close", "paid_up_value", "market_lot", "face_value"]].head())

if unmatched:
    print("Warning: some market records did not match the company master.")
else:
    print("All market records matched the company master.")

In [ ]:
date_features = merged_df[["trade_date"]].copy()
date_features["year"] = date_features["trade_date"].dt.year
date_features["month"] = date_features["trade_date"].dt.month
date_features["day"] = date_features["trade_date"].dt.day
date_features["weekday"] = date_features["trade_date"].dt.day_name()

print("=== DATETIME FEATURES ===")
display(date_features.head())

returns_df = merged_df.sort_values(["nse_symbol", "trade_date"]).copy()
returns_df["daily_return"] = returns_df.groupby("nse_symbol")["close"].pct_change()
returns_df["rolling_3d_close"] = returns_df.groupby("nse_symbol")["close"].transform(lambda x: x.rolling(3).mean())
returns_df["rolling_3d_volatility"] = returns_df.groupby("nse_symbol")["daily_return"].transform(lambda x: x.rolling(3).std())

print("\n=== RETURNS / ROLLING FEATURES ===")
display(returns_df[[
    "nse_symbol", "company_name", "trade_date", "close",
    "daily_return", "rolling_3d_close", "rolling_3d_volatility"
]].head(15))

print(f"Daily-return NaNs: {returns_df['daily_return'].isna().sum():,}")

## Phase 2.4 Completion

Core Pandas and data-handling operations were validated on the project's persisted NSE company-master and historical market data. Final investment targets, historical features, and ML inputs are created in later phases.